In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
import google.generativeai as genai
import os
import time
import pandas as pd
from faker import Faker
import random

In [2]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-1.5-flash")

In [3]:
fake = Faker()

# Function to generate a random engagement level
def random_engagement_level():
    return random.choice(['Low', 'Medium', 'High'])

# Function to generate a random job title
def random_job_title():
    job_titles = ['CEO', 'CFO', 'CTO', 'COO', 'VP of Sales', 'Marketing Manager', 'Product Manager', 'Lead Engineer', 'Business Analyst']
    return random.choice(job_titles)

# Function to generate random client data
def generate_client_data(num_clients=500):
    clients = []
    
    for _ in range(num_clients):
        company_name = fake.company()
        num_key_individuals = random.randint(1, 5)  # Number of key contacts per company (1 to 3 individuals)

        for _ in range(num_key_individuals):
            client = {
                'Company Name': company_name,
                'Company LinkedIn': f"https://www.linkedin.com/company/{company_name.replace(' ', '').lower()}",
                'Company Website': fake.url(),
                'Deals Completed': random.randint(1, 50),
                'Processes Initiated': random.randint(0, 5),
                'Engagement Level': random_engagement_level(),
                'Key Individual Name': fake.name(),
                'Key Individual LinkedIn': f"https://www.linkedin.com/in/{fake.name().replace(' ', '').lower()}",
                'Email': fake.email(),
                'Phone Number': fake.phone_number(),
                'Job Title': random_job_title()  # Replace 'Status' with 'Job Title'
            }
            clients.append(client)

    return clients

# Generate the dummy client data
client_data = generate_client_data()

# Create a pandas DataFrame for better visualization and manipulation
df_clients = pd.DataFrame(client_data)

# Display the first few rows of the dataframe
print(df_clients.head())

# Save the data to a CSV file
df_clients.to_csv('dummy_client_data.csv', index=False)

      Company Name                                Company LinkedIn  \
0  Chavez and Sons  https://www.linkedin.com/company/chavezandsons   
1  Chavez and Sons  https://www.linkedin.com/company/chavezandsons   
2  Chavez and Sons  https://www.linkedin.com/company/chavezandsons   
3    Shelton Group   https://www.linkedin.com/company/sheltongroup   
4    Shelton Group   https://www.linkedin.com/company/sheltongroup   

                    Company Website  Deals Completed  Processes Initiated  \
0                https://meyer.org/               17                    3   
1          https://watson-roth.net/                5                    2   
2                 http://price.com/               25                    2   
3  https://www.garcia-anderson.com/               24                    5   
4                 https://pope.com/               20                    5   

  Engagement Level  Key Individual Name  \
0             High  Samantha Newton DVM   
1             High      Jessic

In [88]:
def ask_gemini_for_insights(prompt_text):
    """
    Interacts with Google Gemini API to extract insights, key contacts, job status, funding details, etc.
    from provided text input.
    """
    try:
        # Make a request to Google Generative AI for insight generation
        response = model.generate_content(
            prompt_text,
            generation_config = genai.GenerationConfig(
                max_output_tokens=300,
                temperature=0.1,
            )
        )
        
        # Return the insights provided by Gemini
        return response.text
    except Exception as e:
        print(f"Error generating insights: {e}")
        return None

In [109]:
df_clients = pd.read_csv('dummy_client_data.csv')

# 1. Extracting company key contacts and funding details
company_name = df_clients.sample()["Company Name"]
sector = "technology"
company_prompt = f"""
        Create a hypothetical comprehensive summary of the key public data for the fictional company {company_name}, specifically focusing on:
        1. Recent deals and acquisitions in the {sector} industry in 2024. Include company names, deal amounts, dates, and the nature of the deals.
        2. Latest fundraising activities and investments related to {company_name} or its competitors in the {sector}. Include the amount raised, investors involved, and the date.
        3. New market entrants in the {sector} that may impact {company_name}. Provide company names and their market positioning in 2024.
        4. Mandates, partnerships, or strategic alliances formed by {company_name} and its competitors in the {sector} since 2024. Include details such as scope and duration of mandates or collaborations.
        Ensure all information is hypothetical, based on the current market trends in {sector}.
        """

# Ask Gemini to process this company information and extract key insights
company_insight = ask_gemini_for_insights(company_prompt)
print(company_insight)

## Hypothetical Public Data Summary: 16 Ramsey, Miller and Braun (2024)

**1. Recent Deals and Acquisitions (Technology Industry, 2024):**

* **Acquisition of "InnovateAI" (March 15th, 2024):** 16 Ramsey, Miller and Braun acquired InnovateAI, a startup specializing in generative AI for marketing automation, for $150 million. This deal significantly expands R&M&B's capabilities in AI-driven marketing solutions.
* **Strategic Investment in "CyberSec Solutions" (June 20th, 2024):** R&M&B invested $30 million in CyberSec Solutions, a cybersecurity firm focusing on cloud-based threat detection. This investment strengthens R&M&B's security infrastructure and provides access to cutting-edge cybersecurity technologies.
* **Competitor Activity:**  "TechGiant Corp" acquired "DataStream Analytics" for $500 million in July, bolstering their big data analytics portfolio. This acquisition poses a potential threat to R&M&B's market share in data analytics.


**2. Latest Fundraising Activities and Inv

In [104]:
df_clients = pd.read_csv('dummy_client_data.csv')